# SimJEB stress surrogate - trainingTrains the MeshGraphNet on the 332 prebuilt brackets.**Before running, in the notebook sidebar:**1. **+ Add Input** -> Datasets -> `simjeb-features`2. **Settings -> Accelerator** -> GPU T4 x2 (only one is used)3. **Settings -> Internet** -> On  (needed for pip and git clone)Then **Run All**, or **Save & Run All (Commit)** to let it finish withoutkeeping the browser tab open.The dataset is read-only and holds the features, the split and the scaling.Nothing is built here - stages 1 and 2 already ran locally.

## 1. The one missing library

In [ ]:
# Kaggle's image has torch but not torch_geometric. Only the pure-Python# part is needed - the model uses torch_geometric.utils.scatter, not the# compiled scatter/sparse extensions - so a plain pip install is enough.!pip install -q torch_geometric

## 2. The code

In [ ]:
# Cloned rather than pasted, so the notebook cannot drift from the repo.!rm -rf /kaggle/working/code!git clone -q https://github.com/Vedavamsi-3/simjeb-stress-surrogate.git /kaggle/working/code!ls /kaggle/working/code

## 3. Where things areThe four variables below are the whole Kaggle adaptation. `config.py` readsthem and falls back to its local paths when they are unset, so no file in therepo needs editing.`/kaggle/input` is read-only, which is why the output goes to`/kaggle/working` - the only writable place, and the only one whose files youcan download afterwards.

In [ ]:
import osimport sysDATASET = "/kaggle/input/simjeb-features"os.environ["SIMJEB_FEATURE_DIR"]  = f"{DATASET}/features"os.environ["SIMJEB_SPLIT_FILE"]   = f"{DATASET}/split.json"os.environ["SIMJEB_SCALING_FILE"] = f"{DATASET}/scaling.npz"os.environ["SIMJEB_OUTPUT_DIR"]   = "/kaggle/working/output"sys.path.insert(0, "/kaggle/working/code")import torchprint("torch      :", torch.__version__)print("cuda        :", torch.cuda.is_available(),      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")print()import configprint(config.describe())

## 4. Does the data look right?Three numbers to check before spending ten hours: 332 feature files, a splitthat adds up to 332, and a scaling that loads. If any of these is wrong, itis wrong now rather than at hour nine.

In [ ]:
from pathlib import Pathfrom simjeb import scaling as scaling_modulefrom simjeb import splits as splits_modulefeature_files = sorted(Path(config.FEATURE_DIR).glob("*.npz"))print(f"feature files : {len(feature_files)}")split = splits_module.Split.load(config.SPLIT_FILE)print(f"split         : {len(split.train)} train, {len(split.val)} val, "      f"{len(split.test)} test  = {len(split.all_ids)}")scalers = scaling_module.Scalers.load(config.SCALING_FILE)print(f"scaling       : {len(scalers.node.mean)} node columns, "      f"{len(scalers.edge.mean)} edge columns")# Every bracket the split names must actually be on disk.on_disk = {int(path.stem) for path in feature_files}missing = [i for i in split.all_ids if i not in on_disk]print(f"missing       : {missing if missing else 'none'}")assert not missing, "the split names brackets that were not uploaded"

## 5. One forward passCheap insurance. Builds the real model, runs one batch on the GPU, and printsthe loss. Ten seconds, and it catches a shape mismatch or a missing librarybefore the long run starts.

In [ ]:
import torchfrom simjeb import batching as batching_modulefrom simjeb import features as features_modulefrom simjeb import model as model_moduledevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")network = model_module.MeshGraphNet(    node_width=features_module.N_NODE_FEATURES,    edge_width=features_module.N_EDGE_FEATURES,    hidden_width=config.HIDDEN_WIDTH,    message_rounds=config.MESSAGE_ROUNDS,    output_width=1,    dropout=config.DROPOUT,).to(device)print(network.describe())loader = batching_module.Loader(    config.FEATURE_DIR, split.train[:config.BATCH_SIZE], scalers,    features_module.load, batch_size=config.BATCH_SIZE, shuffle=False)batch = next(iter(loader)).to(device)prediction = network.predict(batch)loss = torch.nn.functional.mse_loss(prediction, batch.target)print(f"batch      : {batch.n_nodes:,} nodes, "      f"{batch.edge_index.shape[1]:,} directed edges")print(f"prediction : {tuple(prediction.shape)}")print(f"loss       : {float(loss):.4f}")print(f"gpu memory : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"      if torch.cuda.is_available() else "")del network, batch, prediction, losstorch.cuda.empty_cache() if torch.cuda.is_available() else None

## 6. TrainUp to 300 epochs, stopping when the validation loss has not improved for 30.`MAX_HOURS` in `config.py` is 10.5, which leaves margin inside Kaggle's12-hour limit.**Resumable.** Full state is written every epoch to`/kaggle/working/output/runs/full/checkpoint.pt`. If the session dies, committhis notebook as a dataset and point a new session at it, or simply run againin the same session - it carries on from the last epoch rather than startingover.A loss curve is saved at the end, and the epoch table below is printed live.

In [ ]:
!cd /kaggle/working/code && python -u run_3_train.py

## 7. Score it - onceEvery look at the test set during development turns it into a secondvalidation set. The script refuses a second run without `--again` for thatreason.

In [ ]:
!cd /kaggle/working/code && python -u run_4_evaluate.py --run full

## 8. The figures

In [ ]:
!cd /kaggle/working/code && python -u run_5_plots.py --run full

In [ ]:
from IPython.display import Image, displayplots = Path("/kaggle/working/output/runs/full/plots")for name in ("loss_curve.png", "predicted_vs_actual.png",             "per_bracket_error.png", "error_vs_peak_stress.png",             "error_distribution.png"):    path = plots / name    if path.is_file():        print(name)        display(Image(str(path)))

## 9. Take it homeEverything worth keeping is small. The zip below is a few MB and appears inthe notebook's **Output** tab for download.`run_5_plots.py` can redraw every figure from these files on your ownmachine, so there is no reason to come back to a GPU to change a plot.

In [ ]:
!cd /kaggle/working/output && zip -r -q /kaggle/working/simjeb_results.zip runs reports!ls -la /kaggle/working/simjeb_results.zip!unzip -l /kaggle/working/simjeb_results.zip | tail -20